In [ ]:
import pandas as pd
import pymc as pm
import arviz as az
import numpy as np

import sys
sys.path.append("..")
from utils.clean_names import normalize_team_name

In [2]:
import json

# Load full season data
df = pd.read_csv('full_season.csv')

# Load canonical team list from JSON
with open('data/ncaa_teams.json', 'r') as f:
    team_list = json.load(f)

df['team'] = df['team'].apply(normalize_team_name)
df['opp_team'] = df['opp_team'].apply(normalize_team_name)

df = df[df['team'].isin(team_list)]
df = df[df['opp_team'].isin(team_list)]

In [3]:
print(df['team'].unique())

['New Orleans' 'TCU' 'Ole Miss' 'SE Louisiana' 'Delaware' 'Bucknell'
 'Miami' 'Jacksonville' 'Utah Tech' 'South Dakota' 'Air Force' 'Belmont'
 'North Dakota' 'Alabama' 'Fairleigh Dickinson' 'Iowa State'
 'Long Island University' 'Notre Dame' 'Oklahoma' 'St. Francis (PA)'
 'Virginia' 'Rider' 'Holy Cross' 'Providence' 'UConn' 'New Haven'
 'New Hampshire' 'Clemson' 'Georgetown' 'Morgan State' 'Marist' 'Xavier'
 'Canisius' 'Dayton' 'Niagara' 'Duquesne' 'Eastern Washington' 'UCLA'
 'Maine' 'George Washington' 'Loyola Chicago' 'Cleveland State'
 'Rhode Island' 'Stetson' 'Saint Louis' 'Southeast Missouri State'
 'St. Bonaventure' 'Bradley' 'Boston College' 'Florida Atlantic'
 'California' 'Cal State Bakersfield' 'South Carolina State' 'Louisville'
 'NC State' 'North Carolina Central' 'Central Arkansas' 'North Carolina'
 'Pittsburgh' 'Youngstown State' 'SMU' 'Tarleton State' 'Syracuse'
 'Binghamton' 'Virginia Tech' 'Charleston Southern' 'American University'
 'Wake Forest' 'UT Rio Grande Valle

In [4]:
test = pd.read_csv('data/master_df.csv')

In [5]:
df['result'] = np.where(df['team_pts'] > df['opp_pts'], 'W', 'L')
records = (
    df.groupby('team')['result']
      .value_counts()
      .unstack(fill_value=0)
)
df = df.merge(records, on = 'team')

In [6]:
# Encode categorical variables as indices
teams = df['team'].unique()
team_idx = df['team'].apply(lambda x: np.where(teams == x)[0][0])
opponents = df['opp_team'].unique()
opp_idx = df['opp_team'].apply(lambda x: np.where(opponents == x)[0][0])

conditions = [
    df['home'] == True,
    df['away'] == True,
    df['neutral'] == True
]

choices = [1, -1, 0]

df['location'] = np.select(conditions, choices, default=0)

In [ ]:
with pm.Model() as model:
    
    # League average PPP
    mu = pm.Normal('mu', mu=1, sigma=0.5)
    
    # Team random effects (offense)
    sigma_team = pm.HalfNormal('sigma_team', sigma=1.0)
    team_off_raw = pm.Normal('team_off_raw', mu=0, sigma=1, shape=len(teams))
    team_off = pm.Deterministic('team_off', sigma_team * (team_off_raw))
    
    # Opponent random effects (defense)
    sigma_opp = pm.HalfNormal('sigma_opp', sigma=1.0)
    opp_def_raw = pm.Normal('opp_def_raw', mu=0, sigma=1, shape=len(opponents))
    opp_def = pm.Deterministic('opp_def', sigma_opp * (opp_def_raw))
    
    # Home-court effect
    beta_home = pm.Normal('beta_home', mu=0, sigma=0.02)
    
    # Expected PPP for each game
    ppp_hat = (
        mu
        + team_off[team_idx]             # offense of team
        - opp_def[opp_idx]               # defense of opponent
        + beta_home * df['location']         # home-court
    )
    
    # Likelihood
    sigma = pm.HalfNormal('sigma', sigma=0.1)
    y = pm.Normal('y', mu=ppp_hat, sigma=sigma, observed=df['ppp_off_team'])
    
    # Sample posterior
    trace = pm.sample(2000, tune=500, target_accept=0.9, progressbar=True)

    ppc = pm.sample_posterior_predictive(trace, var_names=['y'])

# Examine results
az.summary(trace, var_names=["mu", "team_off", "opp_def", "beta_home"])


c:\Users\rpwju\AppData\Local\Programs\Python\Python313\Lib\site-packages\pymc\model\core.py:1319: ImputationWarning: Data in y contains missing values and will be automatically imputed from the sampling distribution.
  warnings.warn(impute_message, ImputationWarning)
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu, sigma_team, team_off_raw, sigma_opp, opp_def_raw, beta_home, sigma, y_unobserved]


Output()

Sampling 4 chains for 500 tune and 2_000 draw iterations (2_000 + 8_000 draws total) took 48 seconds.


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
mu,1.095,0.005,1.085,1.105,0.0,0.0,1342.0,2436.0,1.0
team_off[0],0.000,0.024,-0.043,0.048,0.0,0.0,8992.0,6021.0,1.0
team_off[1],0.043,0.024,-0.002,0.089,0.0,0.0,11103.0,5387.0,1.0
team_off[2],0.037,0.024,-0.010,0.082,0.0,0.0,8798.0,5500.0,1.0
team_off[3],-0.094,0.025,-0.140,-0.046,0.0,0.0,9486.0,5457.0,1.0
...,...,...,...,...,...,...,...,...,...
opp_def[361],0.000,0.025,-0.045,0.050,0.0,0.0,10016.0,5952.0,1.0
opp_def[362],-0.003,0.026,-0.051,0.047,0.0,0.0,11023.0,5533.0,1.0
opp_def[363],-0.019,0.026,-0.067,0.030,0.0,0.0,11042.0,5157.0,1.0
opp_def[364],-0.011,0.025,-0.056,0.035,0.0,0.0,10095.0,6021.0,1.0


In [13]:
teams = df['team'].unique()

In [14]:
team_idx_map = {team: i for i, team in enumerate(teams)}
team_name = "Virginia Tech"  # example

# Posterior samples for team offense
off_samples = trace.posterior['team_off'].sel(team_off_dim_0=team_idx_map[team_name])

# Posterior mean (expected rating)
off_mean = off_samples.mean().item()

# Adjusted offensive efficiency (points per possession)
mu_mean = trace.posterior['mu'].mean().item()
adj_off_eff = (mu_mean + off_mean)

print(f"{team_name} adjusted offensive efficiency: {adj_off_eff:.2f} points per possession")


Virginia Tech adjusted offensive efficiency: 1.16 points per possession


In [15]:
# Suppose opponents are listed in 'Opponent' variable
opponents = df['opp_team'].unique()
opp_idx_map = {opp: i for i, opp in enumerate(opponents)}

opp_name = "virginia"  # example
def_samples = trace.posterior['opp_def'].sel(opp_def_dim_0=opp_idx_map[opp_name])

def_mean = def_samples.mean().item()
adj_def_eff = (mu_mean - def_mean)

print(f"{opp_name} adjusted defensive efficiency: {adj_def_eff:.2f} points per possession")

KeyError: 'virginia'

In [16]:
adj_off_eff = {}
for team, i in team_idx_map.items():
    off_samples = trace.posterior['team_off'].sel(team_off_dim_0=i)
    adj_off_eff[team] = (trace.posterior['mu'].mean().item() + off_samples.mean().item())

adj_def_eff = {}
for opp, i in opp_idx_map.items():
    def_samples = trace.posterior['opp_def'].sel(opp_def_dim_0=i)
    adj_def_eff[opp] = (trace.posterior['mu'].mean().item() - def_samples.mean().item())

off_df = pd.DataFrame({
    'Team': list(adj_off_eff.keys()),
    'AdjOffEff': list(adj_off_eff.values())
})

def_df = pd.DataFrame({
    'Team': list(adj_def_eff.keys()),
    'AdjDefEff': list(adj_def_eff.values())
})

rankings = off_df.merge(def_df, on="Team", how="left")
rankings = rankings.merge(records, left_on='Team', right_on='team')

rankings['Total'] = rankings['AdjOffEff'] - rankings['AdjDefEff']
rankings = rankings.sort_values('Total', ascending=False)
rankings['Rank'] = range(1, len(rankings) + 1)

print(rankings)

rankings = rankings[['Rank', 'Team', 'W', 'L', 'AdjOffEff', 'AdjDefEff', 'Total']]

rankings.to_csv('data/current_rankings.csv', index = False)

                         Team  AdjOffEff  AdjDefEff   L   W     Total  Rank
221                      Duke   1.275379   0.898980   2  27  0.376399     1
168                  Michigan   1.270799   0.917713   2  27  0.353087     2
182                   Arizona   1.249813   0.922006   2  27  0.327807     3
183                   Florida   1.240910   0.931904   6  23  0.309005     4
193                  Illinois   1.296437   0.995930   7  22  0.300507     5
..                        ...        ...        ...  ..  ..       ...   ...
167          Western Illinois   0.950357   1.203650  26   2 -0.253293   361
266            Delaware State   0.891689   1.145991  20   3 -0.254302   362
189              Gardner-Webb   0.963522   1.220491  27   1 -0.256969   363
106              Coppin State   0.935251   1.207551  22   6 -0.272300   364
179  Mississippi Valley State   0.897889   1.227259  27   2 -0.329370   365

[365 rows x 7 columns]
